<style>
    /* This is only for styling the notebook for HTML export */
.jp-RenderedHTMLCommon {font: 11pt/1.45 "Times New Roman", Times, serif;}
.jp-Notebook { box-sizing: border-box; max-width: 960px; margin: 32px auto; padding: 0 24px; }
.jp-RenderedHTMLCommon p { text-align: justify; }
.jp-RenderedHTMLCommon h1 { font-size: 20pt; text-align: center; }
.jp-RenderedHTMLCommon h2 { font-size: 12pt; text-align: center; font-variant: small-caps; }
.jp-RenderedHTMLCommon h3, .jp-RenderedHTMLCommon h4, .jp-RenderedHTMLCommon h5 { font-size: 11pt; font-style: italic; text-align: left; }
.jp-RenderedHTMLCommon .nb-author { text-align: center; }
.jp-RenderedHTMLCommon .nb-tabcap { font-size: 9pt; text-align: center; font-variant: small-caps; }
.jp-RenderedHTMLCommon .nb-figcap { font-size: 9pt; text-align: left; }
/* These controls apply only to the exported HTML document. */
body.jp-Notebook .nb-section { margin: 0.75rem 0; }
body.jp-Notebook .nb-section > summary {
  display: flex;
  align-items: baseline;
  gap: 0.5rem;
  cursor: pointer;
  list-style: none;
}
body.jp-Notebook .nb-section > summary::-webkit-details-marker { display: none; }
body.jp-Notebook .nb-section > summary::before { content: "\25b6"; }
body.jp-Notebook .nb-section[open] > summary::before { content: "\25bc"; }
body.jp-Notebook .nb-section > summary:focus-visible {
  outline: 2px solid currentColor;
  outline-offset: 2px;
}
body.jp-Notebook .nb-section > summary > .jp-Cell {
  flex: 1;
  min-width: 0;
  margin: 0;
  padding: 0;
}
body.jp-Notebook .nb-section > summary .jp-InputPrompt { display: none; }
body.jp-Notebook .nb-section-content {
  max-height: 32rem;
  overflow-y: auto;
  padding-left: 1rem;
}
</style>

<script>
(function () {
  function addExportSections() {
    const root = document.querySelector("body.jp-Notebook > main");
    if (!root || root.dataset.nbSectionsReady) return;
    root.dataset.nbSectionsReady = "true";
    const cells = Array.from(root.children).filter(cell => cell.classList.contains("jp-Cell"));
    const stack = [];
    for (const cell of cells) {
      const heading = cell.classList.contains("jp-MarkdownCell")
        ? cell.querySelector(".jp-RenderedHTMLCommon > h2, .jp-RenderedHTMLCommon > h3, .jp-RenderedHTMLCommon > h4, .jp-RenderedHTMLCommon > h5, .jp-RenderedHTMLCommon > h6")
        : null;
      if (heading) {
        const level = Number(heading.tagName.substring(1));
        while (stack.length && stack[stack.length - 1].level >= level) stack.pop();
        const parent = stack.length ? stack[stack.length - 1].content : root;
        const section = document.createElement("details");
        section.className = "nb-section";
        section.open = true;
        const summary = document.createElement("summary");
        summary.appendChild(cell);
        const content = document.createElement("div");
        content.className = "nb-section-content";
        content.tabIndex = 0;
        content.setAttribute("role", "region");
        content.setAttribute("aria-label", heading.textContent.trim());
        section.append(summary, content);
        parent.appendChild(section);
        stack.push({ level, content });
      } else {
        (stack.length ? stack[stack.length - 1].content : root).appendChild(cell);
      }
    }
  }
  if (document.readyState === "loading") {
    document.addEventListener("DOMContentLoaded", addExportSections, { once: true });
  } else {
    addExportSections();
  }
})();
</script>

# CSCU9YE Metaheuristics for Multi-Resource Workload Allocation



<p class="nb-author">Student ID: 30065980</p>



## Conceptual Understanding


I write this section only to serve as a descriptive understanding of the assignment before moving on to mathematical modeling and computer modeling.


### What decision do we have to make?


- For each candidate model, we need to determine whether it should be **deployed** on the autonomous edge platform
- Determine which **combination of AI models** should be selected to operate together


### Objective: Maximize total utility


- Functional utility: Maximize the combined utility provided by the selected AI models.
- Model selection: Prioritize combinations of models that provide higher overall utility.
- Resource efficiency: Achieve the highest possible utility without exceeding the available hardware resources.


### Constraints


- Each model must either be selected or excluded
- The total resource consumption of selected models must not exceed the available capacity of each resource.
- The combined consumption of RAM, GPU compute, storage, power, and bandwidth, where applicable, must remain within their respective limit.
- All resource constraints must be satisfied simultaneously for a model selection to be feasible.
- A model combination is considered infeasible if it exceeds even one resource capacity.
- The resource requirements and utility values of each AI model are assumed to be fixed and known.


## Problem Formulation

### Mathematical Model: Hardware-Constrained AI Model Selection

The problem is a binary selection problem: each candidate AI model is either deployed or not deployed.


#### 1. Sets, Indices, and Decision Variables

Let $J$ be the set of candidate AI models and $I$ the set of resource types:

$$
J = \{1,2,\ldots,n\}, \qquad I = \{1,2,\ldots,m\}
$$

Here $j \in J$ identifies a model, $i \in I$ identifies a resource, $n$ is the number of candidate models, and $m$ is the number of resource types.

For every model $j$, define the binary decision variable:

$$
x_j =
\begin{cases}
1 & \text{if model } j \text{ is selected}, \\
0 & \text{otherwise}.
\end{cases}
$$

A complete candidate selection is the binary vector $x=(x_1,x_2,\ldots,x_n)$.


#### 2. Parameters

| Symbol | Definition |
|:---|:---|
| $p_j$ | Functional utility provided by model $j$ |
| $r_{i,j}$ | Consumption of resource $i$ by model $j$ |
| $b_i$ | Capacity available for resource $i$ |
| $n$ | Number of candidate models |
| $m$ | Number of resource constraints |

All values are fixed inputs from the corresponding dataset. The datasets do not label the physical resource units; consequently, this notebook refers to Resource 1, Resource 2, and so forth.

| Input | Instance 1 | Instance 2 |
|:---|---:|---:|
| Models ($n$) | 28 | 90 |
| Resources ($m$) | 2 | 5 |
| Capacity for Resource 1 | 500 | 2100 |
| Capacity for Resource 2 | 500 | 1100 |
| Capacity for Resource 3 | — | 3300 |
| Capacity for Resource 4 | — | 3700 |
| Capacity for Resource 5 | — | 3600 |


#### 3. Objective Function

Maximize the total utility of the selected models:

$$
\max f(x)=\sum_{j=1}^{n}p_j x_j
$$

If a model is selected ($x_j=1$), its utility is included; otherwise, it contributes zero.


#### 4. Constraints

For every resource $i$, the total consumption cannot exceed its capacity:

$$
\sum_{j=1}^{n}r_{i,j}x_j \le b_i, \qquad i=1,\ldots,m
$$

All decisions are binary:

$$
x_j \in \{0,1\}, \qquad j=1,\ldots,n
$$

A selection is **feasible** only if *every* resource limit is satisfied. This is the 0–1 multidimensional knapsack model provided in the assignment specification.


#### 5. Instance 1: 28 Models and 2 Resource Limits

Substituting the dataset dimensions and two capacities into the general model gives:

$$
\begin{aligned}
\text{Maximize} \quad & f(x)=\sum_{j=1}^{28}p_j x_j \\[6pt]
\text{Subject to} \quad & \sum_{j=1}^{28}r_{1,j}x_j \le 500 \\[5pt]
& \sum_{j=1}^{28}r_{2,j}x_j \le 500 \\[5pt]
& x_j \in \{0,1\}, \quad j=1,\ldots,28.
\end{aligned}
$$

Both resource constraints must hold for a feasible selection.


#### 6. Instance 2: 90 Models and 5 Resource Limits

The same model applies with five capacities:

$$
\begin{aligned}
\text{Maximize} \quad & f(x)=\sum_{j=1}^{90}p_j x_j \\[6pt]
\text{Subject to} \quad & \sum_{j=1}^{90}r_{1,j}x_j \le 2100 \\[5pt]
& \sum_{j=1}^{90}r_{2,j}x_j \le 1100 \\[5pt]
& \sum_{j=1}^{90}r_{3,j}x_j \le 3300 \\[5pt]
& \sum_{j=1}^{90}r_{4,j}x_j \le 3700 \\[5pt]
& \sum_{j=1}^{90}r_{5,j}x_j \le 3600 \\[5pt]
& x_j \in \{0,1\}, \quad j=1,\ldots,90.
\end{aligned}
$$

The mathematical model is unchanged between the two instances; only the input values and dimensions differ.


## Computer Modelling

### Data Loading

Before implementing a search algorithm, I load the two supplied workload instances. This gives the optimisation code access to the model utilities, resource demands, and capacity limits.

In [6]:
from pathlib import Path
import numpy as np

np.random.seed(42)

In [ ]:
def load_workload_instance(file_path):
    """
    Loads a hardware-constrained workload allocation dataset.
    
    Parameters:
        file_path (str): Path to the dataset plain text file (.txt).
        
    Returns:
        n (int): Number of candidate AI models.
        m (int): Number of hardware resource constraints.
        utilities (np.ndarray): 1D array of shape (n,) containing model utility scores (p_j).
        demands (np.ndarray): 2D array of shape (m, n) containing resource consumption demands (r_{i,j}).
        capacities (np.ndarray): 1D array of shape (m,) containing hardware capacity limits (b_i).
    """
    with open(file_path, 'r') as f:
        tokens = f.read().split()
        
    if not tokens:
        raise ValueError(f"Dataset file at '{file_path}' is empty or invalid.")
        
    n = int(tokens[0])
    m = int(tokens[1])
    idx = 2
    
    # Model utility scores (length n)
    utilities = np.array([float(x) for x in tokens[idx : idx + n]])
    idx += n
    
    # Resource demand matrix (shape m x n)
    demands = np.zeros((m, n))
    for i in range(m):
        demands[i, :] = [float(x) for x in tokens[idx : idx + n]]
        idx += n
        
    # Maximum hardware capacity limits (length m)
    capacities = np.array([float(x) for x in tokens[idx : idx + m]])
    
    return n, m, utilities, demands, capacities


Each dataset file stores its values in this order: the number of models ($n$), the number of resources ($m$), $n$ utility scores, $m \times n$ resource-demand values, and $m$ capacity limits. The `load_workload_instance` function reads this format and returns those values as $n$, $m$, a utility vector, a demand matrix, and a capacity vector. I have kept the function exactly as it was provided in the assignment, as required. The demand matrix has one row per resource and one column per model, so its shape is $(m, n)$.

In [ ]:
DATA_DIR = Path("Data")
instance1_path = DATA_DIR / "multi_resource_n28_m2.txt"
instance2_path = DATA_DIR / "multi_resource_n90_m5 (1).txt"

for file_path in (instance1_path, instance2_path):
    if not file_path.is_file():
        raise FileNotFoundError(f"Missing dataset file: {file_path}")

n1, m1, p1, R1, b1 = load_workload_instance(instance1_path)
n2, m2, p2, R2, b2 = load_workload_instance(instance2_path)

print("Instance 1 loaded:", n1, "models and", m1, "resources")
print("Instance 2 loaded:", n2, "models and", m2, "resources")

#Instance 1
print("1. Utility scores shape (p_j)     :", p1.shape)
print("1. Resource demands shape (r_ij)  :", R1.shape)
print("1. Capacity limits (b_i)          :", b1)

#Instance 2
print("2. Utility scores shape (p_j)     :", p2.shape)
print("2. Resource demands shape (r_ij)  :", R2.shape)
print("2. Capacity limits (b_i)          :", b2)

Instance 1 loaded: 28 models and 2 resources
Instance 2 loaded: 90 models and 5 resources
Utility scores shape (p_j)     : (28,)
Resource demands shape (r_ij)  : (2, 28)
Capacity limits (b_i)          : [500. 500.]
Utility scores shape (p_j)     : (90,)
Resource demands shape (r_ij)  : (5, 90)
Capacity limits (b_i)          : [2100. 1100. 3300. 3700. 3600.]


The output shows that Instance 1 contains 28 models and 2 resources, while Instance 2 contains 90 models and 5 resources. Utility scores $p_j$ are stored in one-dimensional arrays. Resource demands $r_{i,j}$ are stored in matrices, with one row per resource and one column per model. Their shapes are $(2, 28)$ and $(5, 90)$.

The capacity vector $b_i$ gives the limit for each resource. Instance 1 has capacities of 500 for both resources. Instance 2 has capacities of 2100, 1100, 3300, 3700, and 3600. A feasible selection must stay within every capacity while maximizing the selected models' total utility.

Each model can be selected or excluded, so $n$ models give $2^n$ possible selections. Instance 1 has 28 models, giving:

$$
2^{28} = 268{,}435{,}456
$$

possible selections.

## Part 1: Baseline Discrete Hill-Climbing Local Search


I structured the baseline hill climber around three procedures described by Luke (2013, Section 2.1): initialisation, modification, and assessment and selection. These procedures determine how the algorithm constructs a feasible starting solution, explores alternative model combinations, and decides which improvement to accept. For the modification procedure, I chose bit flips and one-for-one swaps to explore the neighbourhood, using 2 method forms the overall best-improvement strategy to select the next solution.

### 1.2 Initialisation



In [7]:
def random_feasible_start(utilities, demands, capacities):
    """Build a feasible selection by checking models in random order."""
    selection = np.zeros(len(utilities), dtype=np.int8)
    resource_usage = np.zeros(len(capacities), dtype=float)

    for model in np.random.permutation(len(utilities)):
        proposed_usage = resource_usage + demands[:, model]
        if np.all(proposed_usage <= capacities + 1e-9):
            selection[model] = 1
            resource_usage = proposed_usage

    return selection


I use the function `random_feasible_start()` to create a feasible starting selection for hill climbing before the search begins. It starts with all models excluded, then uses `np.random.permutation()` to shuffle their order so models listed first do not always get priority. It adds a model only when its resource demands fit within every capacity. The `1e-9` tolerance allows for tiny floating-point rounding differences, not meaningful extra resource use.

In [9]:
initial_solution = random_feasible_start(p1, R1, b1)
initial_utility = float(p1 @ initial_solution)
initial_resource_usage = R1 @ initial_solution

print("Initial selection:", initial_solution)
print("Initial utility:", initial_utility)
print("Resource use:", initial_resource_usage)
print("Feasible:", np.all(initial_resource_usage <= b1 + 1e-9))

Initial selection: [1 1 0 0 1 0 0 0 1 1 1 1 1 1 0 1 1 1 1 0 0 1 1 1 1 1 1 0]
Initial utility: 66824.0
Resource use: [500. 492.]
Feasible: True


The output shows the initial selection of 28 models. Each `1` marks a selected model, and each `0` marks an excluded model. The selected models have a total utility of 66,824 and use 500 units of Resource 1 and 492 units of Resource 2. These amounts are within the respective capacities of 500 units each, so the starting selection is feasible.

### 1.3 Modification Procedure


In [10]:
def generate_feasible_neighbors(selection, demands, capacities):
    """Yield feasible one-bit flips and one-for-one swaps."""
    selected = np.flatnonzero(selection)
    unselected = np.flatnonzero(selection == 0)

    for model in range(len(selection)):
        candidate = selection.copy()
        candidate[model] = 1 - candidate[model]
        if np.all(demands @ candidate <= capacities + 1e-9):
            yield candidate

    for remove_model in selected:
        for add_model in unselected:
            candidate = selection.copy()
            candidate[remove_model] = 0
            candidate[add_model] = 1
            if np.all(demands @ candidate <= capacities + 1e-9):
                yield candidate


The function creates neighbouring selections using bit flips and one-for-one swaps. `np.flatnonzero()` identifies the selected and unselected models. For a bit flip, the function copies the current selection and changes one value with `candidate[model] = 1 - candidate[model]`, leaving the original selection unchanged. It yields the candidate only when `np.all(demands @ candidate <= capacities + 1e-9)` confirms that every resource limit is satisfied.

For a swap, the function removes one selected model and adds one unselected model. Removing a model frees its resource use, which can make a replacement feasible when adding that model directly would exceed a capacity. Combining swaps and bit-flip require more candidate checks, but allow the search to consider higher-utility replacements.

In [11]:
feasible_neighbors = list(generate_feasible_neighbors(initial_solution, R1, b1))
print("Feasible neighbouring selections:", len(feasible_neighbors))

Feasible neighbouring selections: 56
